# Modelo XGBoost — Fase 6, paso 4: verificación del módulo del modelo final

Este notebook verifica `src/models/xgboost/model.py`, el módulo que fija el modelo final: los hiperparámetros elegidos en `03_tuning.ipynb` (`learning_rate=0.3`, `max_depth=4`, `min_child_weight=1`), `scale_pos_weight` = benignos / ataques de `y_train` (variante C de `02b_ablacion_smote.ipynb`, calculado por el módulo, no fijado a mano) y el umbral elegido en `04_umbral.ipynb` (`UMBRAL_FINAL = 0.87`). Antes de usar el módulo para la evaluación final, tiene que reproducir lo que dieron esos dos notebooks.

Verificaciones, todas en val:

1. Con umbral 0.5: best_iteration 68, FP 13 y FN 3, igual que en `04_umbral.ipynb` (que a su vez reprodujo `03_tuning.ipynb`). Si no coincide, el notebook se detiene.
2. Con `UMBRAL_FINAL`: FP 11 y FN 3 (recall 0.9984, precision 0.9942, F1 0.9963, F2 0.9976), igual que en `04_umbral.ipynb`. Si no coincide, se reporta.
3. El modelo se guarda en `models/xgboost/` y se vuelve a cargar: las probabilidades en val tienen que ser idénticas antes y después, y los metadatos tienen que traer `scale_pos_weight` igual a benignos / ataques de `y_train`, con esos dos conteos. Si no, el notebook se detiene.

**Test no se carga en este notebook.** `X_test` / `y_test` se reservan para la evaluación final del modelo cerrado. Las métricas en val de este notebook son las mismas de 03 y 04, y son optimistas por el mismo motivo: val ya se usó para el early stopping, para elegir los hiperparámetros y para elegir el umbral.

In [1]:
import json
import sys
from pathlib import Path

# Raíz del repo = primera carpeta hacia arriba que contenga src/.
# Así el notebook funciona sin importar desde dónde se abra el kernel.
RAIZ_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ_REPO) not in sys.path:
    sys.path.append(str(RAIZ_REPO))

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import confusion_matrix

from src.evaluation.metrics import evaluate_model
from src.models.xgboost.model import (
    HIPERPARAMETROS,
    MODEL_DIR,
    UMBRAL_FINAL,
    apply_threshold,
    calcular_scale_pos_weight,
    gpu_disponible_para_xgboost,
    load_xgboost_model,
    predict_xgboost_proba,
    save_xgboost_model,
    train_xgboost_model,
)

DATA_DIR = RAIZ_REPO / "data" / "processed" / "xgboost"
print(f"Python:        {sys.executable}")
print(f"Raíz del repo: {RAIZ_REPO}")
print(f"Datos:         {DATA_DIR}")
print(f"Modelos:       {MODEL_DIR}")
print(f"xgboost {xgb.__version__}")

Python:        C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\venv\Scripts\python.exe
Raíz del repo: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids
Datos:         C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\data\processed\xgboost
Modelos:       C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\models\xgboost
xgboost 3.3.0


### Carga de train y val

Solo train y val, igual que en `03_tuning.ipynb` y `04_umbral.ipynb`. Test no se carga.

Se verifica que `X_train.csv` sea el train real (310,529 filas y 8,230 ataques), que val tenga 66,541 filas y 1,884 ataques, y que la matriz tenga 55 features, sin `Destination Port`, `Source Port`, `Protocol` ni `Fwd PSH Flags`, y sin las columnas de metadatos del split (`Timestamp`, `bloque`).

In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["label_binaria"].to_numpy()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["label_binaria"].to_numpy()

assert list(X_train.columns) == list(X_val.columns), "Columnas distintas entre train y val"

N_TRAIN_ESPERADO = 310_529
N_ATAQUES_TRAIN_ESPERADO = 8_230
assert len(X_train) == N_TRAIN_ESPERADO, f"Se esperaban {N_TRAIN_ESPERADO:,} filas en el train real, hay {len(X_train):,}"
assert y_train.sum() == N_ATAQUES_TRAIN_ESPERADO, (
    f"Se esperaban {N_ATAQUES_TRAIN_ESPERADO:,} ataques en el train real, hay {y_train.sum():,}"
)

N_VAL_ESPERADO = 66_541
N_ATAQUES_VAL_ESPERADO = 1_884
assert len(X_val) == N_VAL_ESPERADO, f"Se esperaban {N_VAL_ESPERADO:,} filas en val, hay {len(X_val):,}"
assert y_val.sum() == N_ATAQUES_VAL_ESPERADO, f"Se esperaban {N_ATAQUES_VAL_ESPERADO:,} ataques en val, hay {y_val.sum():,}"

N_FEATURES_ESPERADO = 55
assert X_train.shape[1] == N_FEATURES_ESPERADO, f"Se esperaban {N_FEATURES_ESPERADO} features, hay {X_train.shape[1]}"

COLUMNAS_EXCLUIDAS = ["Destination Port", "Source Port", "Protocol", "Fwd PSH Flags", "Timestamp", "bloque"]
presentes = [c for c in COLUMNAS_EXCLUIDAS if c in X_train.columns]
assert not presentes, f"Columnas que no deberían estar en X_train/X_val: {presentes}"

for nombre, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
    print(f"{nombre:>5}: X {X.shape} | y {y.shape} | ataques: {y.sum()} ({y.mean():.4%})")

train: X (310529, 55) | y (310529,) | ataques: 8230 (2.6503%)
  val: X (66541, 55) | y (66541,) | ataques: 1884 (2.8313%)


### Entrenamiento con el módulo y verificación contra `03_tuning.ipynb`

`train_xgboost_model` entrena con `HIPERPARAMETROS` y `scale_pos_weight` (calculado de `y_train` con `calcular_scale_pos_weight`) sobre `X_train` completo y early stopping en val, y devuelve el modelo ya pasado a CPU. El dispositivo se elige con la misma verificación que en `02_modelo.ipynb` y `03_tuning.ipynb`: GPU (`device="cuda"`) si un fit de prueba funciona, CPU si no.

Con umbral 0.5 tiene que dar best_iteration 68, FP 13 y FN 3, los valores de `04_umbral.ipynb`. Si no, el notebook se detiene acá: el módulo no estaría reproduciendo el modelo elegido en 03 y usado en 04.

In [3]:
DEVICE = "cuda" if gpu_disponible_para_xgboost() else "cpu"
print(f"Dispositivo para entrenar: {DEVICE}")
print(f"Hiperparámetros:           {HIPERPARAMETROS}")
PESOS = calcular_scale_pos_weight(y_train)
print(f"scale_pos_weight:          {PESOS['n_benignos_train']:,} / {PESOS['n_ataques_train']:,} = "
      f"{PESOS['scale_pos_weight']:.4f}")

modelo, tiempo_entrenamiento = train_xgboost_model(X_train, y_train, X_val, y_val, device=DEVICE)
proba_val = predict_xgboost_proba(modelo, X_val)


def conteos_val(proba: np.ndarray, umbral: float) -> dict:
    """Matriz de confusión en val para un umbral, con apply_threshold del módulo."""
    tn, fp, fn, tp = confusion_matrix(y_val, apply_threshold(proba, umbral), labels=[0, 1]).ravel()
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}


# Valores de 04_umbral.ipynb (reproducen la combinación elegida en 03_tuning.ipynb)
REFERENCIA_03 = {"best_iteration": 68, "FP": 13, "FN": 3}
c_05 = conteos_val(proba_val, 0.5)
obtenido_03 = {"best_iteration": int(modelo.best_iteration), "FP": c_05["FP"], "FN": c_05["FN"]}
print(f"\nTiempo de entrenamiento: {tiempo_entrenamiento:.2f}s")
print(f"Esperado (03/04):     {REFERENCIA_03}")
print(f"Obtenido:             {obtenido_03}")

if obtenido_03 != REFERENCIA_03:
    raise RuntimeError(
        "NO REPRODUCE 03_tuning.ipynb / 04_umbral.ipynb: el modelo entrenado con el módulo no coincide con el elegido. "
        "Se detiene el notebook."
    )
print("Reproduce 03_tuning.ipynb y 04_umbral.ipynb.")

Dispositivo para entrenar: cuda
Hiperparámetros:           {'learning_rate': 0.3, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 1000, 'early_stopping_rounds': 50, 'eval_metric': 'aucpr', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1}
scale_pos_weight:          302,299 / 8,230 = 36.7313



Tiempo de entrenamiento: 1.02s
Esperado (03/04):     {'best_iteration': 68, 'FP': 13, 'FN': 3}
Obtenido:             {'best_iteration': 68, 'FP': 13, 'FN': 3}
Reproduce 03_tuning.ipynb y 04_umbral.ipynb.


### Umbral final en val y verificación contra `04_umbral.ipynb`

Con `UMBRAL_FINAL` (0.87) tiene que dar FP 11 y FN 3, con recall 0.9984, precision 0.9942, F1 0.9963 y F2 0.9976. FP y FN se comparan exactos; las métricas, redondeadas a 4 decimales, que es como se reportaron en 04. Si algo no coincide, se reporta acá sin detener el notebook.

`evaluate_model` con `save_dir=None`: no escribe `reports/metrics/`, porque esto es val, no la evaluación final en test.

In [4]:
y_pred_final = apply_threshold(proba_val)  # UMBRAL_FINAL por defecto
resultados_final = evaluate_model(
    y_val,
    y_pred_final,
    y_proba=proba_val,
    model_name=f"xgboost_final_umbral_{UMBRAL_FINAL:g}_val",
    training_time_seconds=tiempo_entrenamiento,
    save_dir=None,
)

REFERENCIA_04 = {"FP": 11, "FN": 3, "recall": 0.9984, "precision": 0.9942, "f1": 0.9963, "f2": 0.9976}
mc = resultados_final["matriz_confusion"]
obtenido_04 = {
    "FP": mc["falsos_positivos"],
    "FN": mc["falsos_negativos"],
    "recall": round(float(resultados_final["recall"]), 4),
    "precision": round(float(resultados_final["precision"]), 4),
    "f1": round(float(resultados_final["f1_score"]), 4),
    "f2": round(float(resultados_final["f2_score"]), 4),
}
print(f"\nUmbral final: {UMBRAL_FINAL:g}")
print(f"Esperado (04_umbral): {REFERENCIA_04}")
print(f"Obtenido:             {obtenido_04}")

diferencias_04 = {k: (esperado, obtenido_04[k]) for k, esperado in REFERENCIA_04.items() if obtenido_04[k] != esperado}
if diferencias_04:
    print("NO COINCIDE con 04_umbral.ipynb. Diferencias (esperado, obtenido):")
    for k, (esperado, obtenido) in diferencias_04.items():
        print(f"  {k}: {esperado} vs {obtenido}")
else:
    print("Coincide con 04_umbral.ipynb.")


=== Resultados: xgboost_final_umbral_0.87_val ===
Accuracy:  0.9998
Precision: 0.9942
Recall:    0.9984  <- métrica principal
F1-score:  0.9963
F2-score:  0.9976
ROC-AUC:   1.0000
Matriz de confusión -> TN: 64646, FP: 11, FN: 3, TP: 1881
Tiempo de entrenamiento: 1.02s

Umbral final: 0.87
Esperado (04_umbral): {'FP': 11, 'FN': 3, 'recall': 0.9984, 'precision': 0.9942, 'f1': 0.9963, 'f2': 0.9976}
Obtenido:             {'FP': 11, 'FN': 3, 'recall': 0.9984, 'precision': 0.9942, 'f1': 0.9963, 'f2': 0.9976}
Coincide con 04_umbral.ipynb.


### Guardado y carga del modelo

`save_xgboost_model` escribe dos archivos en `models/xgboost/`, en la raíz del repo y fuera de `src/`: el modelo en formato JSON de XGBoost y un JSON con el umbral, los hiperparámetros, `scale_pos_weight` con los conteos de `y_train` de los que sale (`n_benignos_train`, `n_ataques_train`) y best_iteration. `save_xgboost_model` recibe `y_train` y, antes de escribir, verifica que el `scale_pos_weight` del modelo sea exactamente benignos / ataques. Si ya existen, se sobrescriben.

Después se cargan con `load_xgboost_model` y se comparan con el modelo entrenado. Las probabilidades en val tienen que ser idénticas (`np.array_equal`, sin tolerancia), best_iteration tiene que ser el mismo, y los metadatos tienen que coincidir con el módulo. `scale_pos_weight` se verifica tres veces: en los metadatos (igual a `n_benignos_train / n_ataques_train`, y esos conteos iguales a los de `y_train`), en el modelo entrenado (`get_params()`) y en el booster cargado (su configuración lo guarda en float32, así que se compara en float32). Si algo no coincide, el notebook se detiene.

In [5]:
ruta_modelo, ruta_metadatos = save_xgboost_model(modelo, y_train=y_train)
print(f"Modelo:    {ruta_modelo.relative_to(RAIZ_REPO)} ({ruta_modelo.stat().st_size / 1e6:.2f} MB)")
print(f"Metadatos: {ruta_metadatos.relative_to(RAIZ_REPO)}")
print(ruta_metadatos.read_text(encoding="utf-8"))

modelo_cargado, metadatos = load_xgboost_model()
proba_val_cargado = predict_xgboost_proba(modelo_cargado, X_val)

identicas = np.array_equal(proba_val, proba_val_cargado)
print(f"\nProbabilidades en val idénticas antes y después de guardar: {identicas} "
      f"(diferencia absoluta máxima: {np.abs(proba_val - proba_val_cargado).max():g})")
print(f"best_iteration: entrenado {modelo.best_iteration} | cargado {modelo_cargado.best_iteration} | "
      f"metadatos {metadatos['best_iteration']}")

problemas = []
if not identicas:
    problemas.append("las probabilidades en val cambian")
if modelo_cargado.best_iteration != modelo.best_iteration:
    problemas.append(f"best_iteration del modelo cargado {modelo_cargado.best_iteration} != {modelo.best_iteration}")
if metadatos["best_iteration"] != modelo.best_iteration:
    problemas.append(f"best_iteration de los metadatos {metadatos['best_iteration']} != {modelo.best_iteration}")
if metadatos["umbral"] != UMBRAL_FINAL:
    problemas.append(f"umbral de los metadatos {metadatos['umbral']} != {UMBRAL_FINAL}")
if metadatos["hiperparametros"] != HIPERPARAMETROS:
    problemas.append(f"hiperparámetros de los metadatos {metadatos['hiperparametros']} != {HIPERPARAMETROS}")
for clave in ("scale_pos_weight", "n_benignos_train", "n_ataques_train"):
    if metadatos.get(clave) != PESOS[clave]:
        problemas.append(f"{clave} de los metadatos {metadatos.get(clave)} != {PESOS[clave]} (calculado de y_train)")
if metadatos.get("scale_pos_weight") != metadatos.get("n_benignos_train", 0) / metadatos.get("n_ataques_train", 1):
    problemas.append("scale_pos_weight de los metadatos != n_benignos_train / n_ataques_train")
if modelo.get_params()["scale_pos_weight"] != PESOS["scale_pos_weight"]:
    problemas.append(f"scale_pos_weight del modelo entrenado {modelo.get_params()['scale_pos_weight']} != {PESOS['scale_pos_weight']}")
spw_booster = json.loads(modelo_cargado.get_booster().save_config())["learner"]["objective"]["reg_loss_param"]["scale_pos_weight"]
print(f"scale_pos_weight: metadatos {metadatos['scale_pos_weight']} | booster cargado {spw_booster} (float32)")
if np.float32(float(spw_booster)) != np.float32(PESOS["scale_pos_weight"]):
    problemas.append(f"scale_pos_weight del booster cargado {spw_booster} != {PESOS['scale_pos_weight']} en float32")

if problemas:
    raise RuntimeError("El modelo cargado no coincide con el entrenado: " + "; ".join(problemas))
print("El modelo cargado coincide con el entrenado.")

Modelo:    models\xgboost\xgboost_final.json (0.19 MB)
Metadatos: models\xgboost\xgboost_final_metadatos.json
{
  "umbral": 0.87,
  "hiperparametros": {
    "learning_rate": 0.3,
    "max_depth": 4,
    "min_child_weight": 1,
    "n_estimators": 1000,
    "early_stopping_rounds": 50,
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
  },
  "scale_pos_weight": 36.73134872417983,
  "n_benignos_train": 302299,
  "n_ataques_train": 8230,
  "best_iteration": 68
}

Probabilidades en val idénticas antes y después de guardar: True (diferencia absoluta máxima: 0)
best_iteration: entrenado 68 | cargado 68 | metadatos 68
scale_pos_weight: metadatos 36.73134872417983 | booster cargado 36.7313499 (float32)
El modelo cargado coincide con el entrenado.


### Resumen

In [6]:
print("=" * 64)
print("RESUMEN — Fase 6, paso 4: verificación del módulo XGBoost (solo val)")
print("=" * 64)
print(f"Módulo:                 src/models/xgboost/model.py | dispositivo {DEVICE}")
print(f"Modelo:                 lr {HIPERPARAMETROS['learning_rate']} | max_depth {HIPERPARAMETROS['max_depth']} | "
      f"min_child_weight {HIPERPARAMETROS['min_child_weight']} | best_iteration {modelo.best_iteration}")
print(f"scale_pos_weight:       {PESOS['scale_pos_weight']:.4f} = {PESOS['n_benignos_train']:,} / "
      f"{PESOS['n_ataques_train']:,} (de y_train; guardado y verificado en los metadatos)")
print(f"Reproduce 03/04:        sí (umbral 0.5: best_iteration {obtenido_03['best_iteration']}, FP {obtenido_03['FP']}, FN {obtenido_03['FN']})")
print(f"Coincide con 04_umbral: {'sí' if not diferencias_04 else 'NO, ver verificación'} "
      f"(umbral {UMBRAL_FINAL:g}: FP {obtenido_04['FP']}, FN {obtenido_04['FN']}, recall {obtenido_04['recall']:.4f}, "
      f"precision {obtenido_04['precision']:.4f}, F1 {obtenido_04['f1']:.4f}, F2 {obtenido_04['f2']:.4f})")
print(f"Guardado y carga:       probabilidades en val idénticas ({ruta_modelo.relative_to(RAIZ_REPO)})")
print("Test: no se cargó. Se reserva para la evaluación final del modelo cerrado.")

RESUMEN — Fase 6, paso 4: verificación del módulo XGBoost (solo val)
Módulo:                 src/models/xgboost/model.py | dispositivo cuda
Modelo:                 lr 0.3 | max_depth 4 | min_child_weight 1 | best_iteration 68
scale_pos_weight:       36.7313 = 302,299 / 8,230 (de y_train; guardado y verificado en los metadatos)
Reproduce 03/04:        sí (umbral 0.5: best_iteration 68, FP 13, FN 3)
Coincide con 04_umbral: sí (umbral 0.87: FP 11, FN 3, recall 0.9984, precision 0.9942, F1 0.9963, F2 0.9976)
Guardado y carga:       probabilidades en val idénticas (models\xgboost\xgboost_final.json)
Test: no se cargó. Se reserva para la evaluación final del modelo cerrado.
